# Assignemnt2 
## 40 different house electricity consumption
There are two datasets **Main_data** and corresponding **Weather_data** 

### What the datasets represent

A synthetic smart-grid panel time series:
40 buildings (meter_id = M001…M040)

Hourly readings for 500 hours starting 2025-01-01 00:00

Total rows: 40 × 500 = 20,000

-------------------
### Main_data
Each row in **Main_data** file is one meter at one hour, with building attributes + pricing + behavior driving energy consumption.

Column groups (what’s inside)
1) Identifiers & time meter_id, timestamp
hour

2) Building “static” attributes (vary by meter)
building_type (residential/commercial/industrial)
region
floor_area_m2
insulation_rating (scaled from 0 to 1)
hvac_age_years (scaled from 0 to 20.2 years)
solar_kw_installed (scaled from 0 to 6.5 kw)
ev_charger (0/1)

3) grid conditions (vary by hour and region)

grid_price_usd_per_kwh
price depends on tariff tier and “temperature stress.”

4) Behavior & generation
   
occupancy_index (0–1 synthetic activity proxy)
solar_generation_kwh (daytime profile, reduced when precip > 0)

5) targeted variable
consumption_kwh (net kWh after solar)
----------------
### Weather_data
Weather data related to each meter_id is available in **Weather_data.CSV** file
Columns inside 
temp_c, humidity_pct, wind_ms, precip_mm
Weather has a diurnal cycle and region baselines; 
Weathers has a diurnal cycle and region baselines; price depends on tariff tier and “temperature stress.”


---------------------
# Please do the following tasks
1) Read the datasets and check the info, type of data, and so on.
2) Generate calendar features such as hour, day_of_week, is_weekend, and is_holiday? **Tips**:  Check the type of 'timestamp'column.
3) Generate per‑meter lag features such as lag1,lag24 over the consumption column. **Tips**: groupby method matters for each meter_id.
4) Generate per‑meter rolling features such as 24-hour consumption average and std.
5) Generate one main dataset with the index of timeseries data and concatenated with the corresponding weather dataset.
   
   --------------------
**From now on, it is very subjective and depends on the usecase you choose. It is up to you to test and check your learning by answering the following questions with different tools and tests.**

6) Choose one arbitrary meter_id related to residential building type and start presenting descriptive statistics over interesting columns such as consumption, weather data, and so on.
7) Please use different plot types visualization tools in seaborn or other Python packages to present more statistical information related to the selected meter_id over targeted column energy consumption or other interesting predictors such as weather data, solar generation, and so on.
8) Study the relation of the user_behavior information, such as weekend, is_holiday, with respect to the targeted variable, energy consumption.
9) How did your targeted variable respond to different energy prices and Tarif_tier?
10) Check that your data are stationary. Is there seasonality or a periodic component in your data? **Tips:** You can just do these analyses over the targeted variable. 
11)  Decompose the main component if there are periodic and nonstationary.
12)  Stationerise your data if it is needed.
----------------------------------------------------------
If you have time, choose another meter_id related to a commercial building and do the tasks from 6 to 10, and compare your findings with the residential one. 

### Task 1 - Read datasets and inspect them
In this step we load the two CSV files, parse the timestamp column, and check shape, sample rows, data types, and full dataframe info.

In [ ]:
from pathlib import Path
import pandas as pd

base_dir = Path('.')
main_path = base_dir / 'Main_data.csv'
weather_path = base_dir / 'Weather_data.CSV'

if not main_path.exists() or not weather_path.exists():
    raise FileNotFoundError(
        f"Expected files not found in {base_dir.resolve()}: Main_data.csv and Weather_data.CSV"
    )

main_df = pd.read_csv(main_path, parse_dates=['timestamp'])
weather_df = pd.read_csv(weather_path, parse_dates=['timestamp'])

print(f'Main data shape: {main_df.shape}')
print(f'Weather data shape: {weather_df.shape}')
display(main_df.head(3))
display(weather_df.head(3))

print('\nMain data dtypes:')
display(main_df.dtypes.to_frame('dtype'))
print('\nWeather data dtypes:')
display(weather_df.dtypes.to_frame('dtype'))

print('\nMain data info:')
main_df.info()
print('\nWeather data info:')
weather_df.info()


### Task 2 - Create calendar features
Here we derive time-based features from `timestamp`: hour, day of week, weekend flag, and a holiday flag.

In [ ]:
from pandas.tseries.holiday import USFederalHolidayCalendar

if 'timestamp' not in main_df.columns:
    raise KeyError("'timestamp' column is required in Main_data")

main_df['hour'] = main_df['timestamp'].dt.hour
main_df['day_of_week'] = main_df['timestamp'].dt.dayofweek
main_df['is_weekend'] = (main_df['day_of_week'] >= 5).astype(int)

cal = USFederalHolidayCalendar()
holidays = cal.holidays(
    start=main_df['timestamp'].min().normalize(),
    end=main_df['timestamp'].max().normalize()
)
main_df['is_holiday'] = main_df['timestamp'].dt.normalize().isin(holidays).astype(int)

display(main_df[['timestamp', 'hour', 'day_of_week', 'is_weekend', 'is_holiday']].head(10))


### Task 3 - Create per-meter lag features
We sort by meter and time, then create lag values of consumption (`lag1` and `lag24`) within each `meter_id` group.

In [ ]:
required_cols = {'meter_id', 'consumption_kwh'}
missing = required_cols - set(main_df.columns)
if missing:
    raise KeyError(f"Missing required columns in Main_data: {missing}")

main_df = main_df.sort_values(['meter_id', 'timestamp']).copy()
grp = main_df.groupby('meter_id', sort=False)['consumption_kwh']
main_df['lag1'] = grp.shift(1)
main_df['lag24'] = grp.shift(24)

display(main_df[['meter_id', 'timestamp', 'consumption_kwh', 'lag1', 'lag24']].head(30))


### Task 4 - Create per-meter rolling features
In this step we calculate 24-hour rolling mean and standard deviation of consumption for each meter separately.

In [ ]:
main_df['rolling24_mean'] = grp.transform(lambda s: s.rolling(window=24, min_periods=1).mean())
main_df['rolling24_std'] = grp.transform(lambda s: s.rolling(window=24, min_periods=1).std())

display(main_df[['meter_id', 'timestamp', 'consumption_kwh', 'rolling24_mean', 'rolling24_std']].head(30))


### Task 5 - Build one main time-series dataset and merge weather
Finally, we join engineered main data with weather data on `meter_id` and `timestamp`, set `timestamp` as index, and save the result.

In [ ]:
if set(['meter_id', 'timestamp']).issubset(weather_df.columns):
    final_df = main_df.merge(weather_df, on=['meter_id', 'timestamp'], how='left', suffixes=('', '_weather'))
elif 'timestamp' in weather_df.columns:
    final_df = main_df.merge(weather_df, on='timestamp', how='left', suffixes=('', '_weather'))
    print('Warning: Weather data has no meter_id; merged only on timestamp.')
else:
    raise KeyError("Weather_data must contain at least 'timestamp', preferably ['meter_id', 'timestamp']")

final_df = final_df.sort_values(['meter_id', 'timestamp']).set_index('timestamp')

print('Final dataset shape:', final_df.shape)
print('Rows preserved after merge:', len(final_df) == len(main_df))
display(final_df.head(10))

output_path = base_dir / 'main_with_weather_features.csv'
final_df.to_csv(output_path)
print(f'Engineered dataset saved to: {output_path.resolve()}')
